In [1]:
import sys
import os


sys.path.append(os.path.abspath(".."))

In [2]:
import gradio as gr
import json
from datetime import datetime
from pymongo import MongoClient
from langchain_core.messages import HumanMessage

from graph.workflow import app
from agents.utils.checklist_format import checklist_format
from agents.utils.generate_pdf import generate_pdf

c:\Users\Lenovo\Desktop\QualiChainAI\Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6078.00it/s]


In [3]:
client = MongoClient("mongodb://localhost:27017/")

db = client["qualichainAI"]

checklists_collection = db["audit_checklists"]
reports_collection = db["audits_reports"]

In [4]:
AUDIT_TYPES = [
    "Audit transport pharmaceutique",
    "Audit système qualité",
    "Audit conformité réglementaire",
    "Audit fournisseur pharmaceutique",
    "Audit entrepôt de stockage pharmaceutique",
    "Audit distributeur pharmaceutique",
    "Audit chaîne du froid pharmaceutique"
]


SITES_TUNISIE = [
    "Tunis Centre",
    "Tunis",
    "Ariana",
    "Ben Arous",
    "Manouba",
    "Nabeul",
    "Sousse",
    "Monastir",
    "Sfax",
    "Bizerte",
    "Gabès",
    "Kairouan",
    "Gafsa",
    "Médenine",
    "Djerba"
]

In [5]:
def generate_checklist(type_audit, site_audit):

    if not type_audit:
        yield "Veuillez sélectionner un type d'audit.", None
        return

    if not site_audit:
        yield "Veuillez sélectionner un site.", None
        return

    yield "⏳ Checklist en cours de génération...", None

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        f"Génère une checklist pour un audit "
                        f"de type {type_audit} "
                        f"pour le site {site_audit}"
                    )
                )
            ]
        }
    )

    content = result["messages"][-1].content

    try:

        checklist = json.loads(content)

    except json.JSONDecodeError:

        yield "❌ Erreur : la checklist retournée n'est pas un JSON valide.", None
        return

    checklist = checklist_format(checklist)

    checklist_id = (
        f"CHK-{checklists_collection.count_documents({}) + 1:03d}"
    )

    checklist = {
        "checklist_id": checklist_id,
        **checklist
    }


    checklists_collection.insert_one(checklist)
    yield (
        f"✅ Checklist {checklist_id} générée avec succès.",
        checklist
    )



In [6]:
def save_completed_checklist(checklist,responsable,*values):

    if not checklist:
        return "❌ Aucune checklist chargée."
    checklist["responsable"] = responsable or ""

    index = 0

    for section in checklist.get("sections", []):

        for point in section.get("points_controle", []):

            resultat = values[index]
            commentaire = values[index + 1]

            if resultat == "Oui":
                point["resultat"] = "Conforme"

            elif resultat == "Non":
                point["resultat"] = "Non conforme"

            else:
                point["resultat"] = ""


            point["commentaire"] = commentaire or ""

            index += 2

    checklists_collection.update_one(
        {
            "checklist_id": checklist["checklist_id"]
        },
        {
            "$set": {
                "responsable": checklist["responsable"],
                "sections": checklist["sections"],
                "status": "remplie"
            }
        }
    )

    return (
        f"✅ Checklist {checklist['checklist_id']} "
        f"enregistrée avec succès."
    )

In [7]:
def export_checklist_pdf(checklist):

    if not checklist:
        raise Exception("Aucune checklist chargée")

    document = checklists_collection.find_one(
        {
            "checklist_id": checklist["checklist_id"]
        }
    )

    if not document:
        raise Exception("Checklist introuvable")

    output_path = (
        f"checklists/{checklist['checklist_id']}.pdf"
    )

    generate_pdf(
        data=document,
        output_file=output_path,
        title=f"Checklist {checklist['checklist_id']}"
    )

    return output_path

In [8]:
def analyze_checklist_ui(checklist):

    if not checklist:
        yield "❌ Aucune checklist chargée."
        return

    yield "⏳ Analyse en cours..."

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Analyse la checklist {checklist['checklist_id']}"
                )
            ]
        }
    )

    try:
        analysis = json.loads(
            result["messages"][-1].content
        )

    except Exception:
        yield "❌ L'analyse retournée n'est pas un JSON valide."
        return

    checklists_collection.update_one(
        {
            "checklist_id": checklist["checklist_id"]
        },
        {
            "$set": {
                "analysis": analysis
            }
        }
    )

    observations = analysis.get("observations", [])
    recommandations = analysis.get("recommandations", [])

    obs_md = "\n".join(f"- {obs}"for obs in observations ) if observations else "Aucune observation."

    reco_md = "\n".join(f"- {rec}"for rec in recommandations) if recommandations else "Aucune recommandation."

    yield f"""
## 📊 Analyse de la checklist

### 🎯 Score de conformité

**{analysis.get("score_conformite", 0)} %**

### 📌 Statut global

**{analysis.get("statut_global", "Non défini")}**

### 📋 Résumé

- **Total des points :** {analysis.get("resume", {}).get("total_points", 0)}
- **Conformes :** {analysis.get("resume", {}).get("conformes", 0)}
- **Non conformes :** {analysis.get("resume", {}).get("non_conformes", 0)}
- **Partiellement conformes :** {analysis.get("resume", {}).get("partiellement_conformes", 0)}
- **Non applicables :** {analysis.get("resume", {}).get("non_applicables", 0)}

### 👁️ Observations

{obs_md}

### 💡 Recommandations

{reco_md}
"""


In [9]:
def generate_report_ui(checklist):

    if not checklist:
        return "❌ Aucune checklist chargée."

    result = app.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Génère un rapport d'audit pour la checklist {checklist['checklist_id']}"
                )
            ]
        }
    )

    report = json.loads(result["messages"][-1].content)

    # sauvegarde MongoDB
    reports_collection.update_one(
        {   
            "checklist_id": checklist["checklist_id"]
        },
        {
            "$set": {
                "report": report
            }
        },
        upsert=True
    )

    document = reports_collection.find_one(
        {
            "checklist_id": checklist["checklist_id"]
        }
    )

    if not document:
        raise Exception("Rapport introuvable")

    output_path = (
        f"reports/{checklist['checklist_id']}_report.pdf"
    )
    generate_pdf(
        data=document,
        output_file=output_path,
        title=f"Rapport d'audit {checklist['checklist_id']} "
    )

    return output_path

In [10]:


with gr.Blocks(title="QualiChain AI") as demo:


    checklist_state = gr.State(None)

    # ========================================================
    # PAGE 1 : CHATBOT
    # ========================================================

    with gr.Tab("💬 Assistant QualiChain AI"):

        gr.Markdown(
            """
            # 🤖 QualiChain AI

            Assistant intelligent pour la conformité pharmaceutique
            GDP/BPD.
            """
        )

        chatbot = gr.Chatbot(
            label="Assistant",
   
        )

        chat_input = gr.Textbox(
            label="Votre question",
            placeholder="Posez votre question..."
        )

        chat_button = gr.Button(
            "Envoyer",
            variant="primary"
        )

        def chatbot_response(message, history):

            result = app.invoke(
                {
                    "messages": [
                        HumanMessage(content=message)
                    ]
                }
            )

            answer = result["messages"][-1].content

            history.append(
                {
                    "role": "user",
                    "content": message
                }
            )

            history.append(
                {
                    "role": "assistant",
                    "content": answer
                }
            )

            return "", history

        chat_button.click(
            chatbot_response,
            inputs=[chat_input, chatbot],
            outputs=[chat_input, chatbot]
        )

        chat_input.submit(
            chatbot_response,
            inputs=[chat_input, chatbot],
            outputs=[chat_input, chatbot]
        )

    # ========================================================
    # PAGE 2 : AUDIT
    # ========================================================

    with gr.Tab("📋 Audit"):

        gr.Markdown(
            """
            # 📋 Création d'une checklist d'audit

            Sélectionnez le type d'audit et le site audité.
            """
        )

        with gr.Row():

            type_audit = gr.Dropdown(
                choices=AUDIT_TYPES,
                label="Type d'audit",
                value=None
            )

            site_audit = gr.Dropdown(
                choices=SITES_TUNISIE,
                label="Site audité",
                value=None,
                allow_custom_value=True
            )

        generate_button = gr.Button(
            "🚀 Créer checklist",
            variant="primary"
        )

        generation_status = gr.Markdown()

        # ====================================================
        # AFFICHAGE DYNAMIQUE DE LA CHECKLIST
        # ====================================================

        @gr.render(inputs=checklist_state)
        def render_checklist(checklist):

            if not checklist:
                return

            gr.Markdown(
                f"""
                ## Checklist : {checklist['checklist_id']}

                **Type :** {checklist['type_audit']}  
                **Site :** {checklist['site_audit']}  
                **Date de création :** {checklist['date_creation']}
                """
            )

            # =================================================
            # RESPONSABLE
            # =================================================

            responsable = gr.Textbox(
                label="Responsable de l'audit",
                placeholder="Nom du responsable..."
            )

            dynamic_values = []

            # =================================================
            # SECTIONS
            # =================================================

            for section in checklist.get("sections", []):

                gr.Markdown(
                    f"## {section['nom_section']}"
                )

                # =============================================
                # QUESTIONS
                # =============================================

                for i, point in enumerate(
                    section.get("points_controle", [])
                ):

                    gr.Markdown(
                        f"""
                        ### Question {i + 1}

                        **Question :**  
                        {point['question']}

                        **Criticité :** `{point['criticite']}`

                        **Preuve attendue :**  
                        {point['preuve_attendue']}
                        """
                    )

                    resultat = gr.Radio(
                        choices=["Oui", "Non"],
                        label="Résultat",
                        value=None
                    )

                    commentaire = gr.Textbox(
                        label="Commentaire",
                        placeholder="Ajouter un commentaire...",
                        lines=2
                    )

                    dynamic_values.append(resultat)
                    dynamic_values.append(commentaire)

                    gr.Markdown("---")

            # =================================================
            # BOUTON ENREGISTRER
            # =================================================

            save_button = gr.Button(
                "💾 Enregistrer checklist",
                variant="primary"
            )

            save_status = gr.Markdown()
            save_button.click(
                            save_completed_checklist,
                            inputs=[
                                checklist_state,
                                responsable,
                                *dynamic_values
                            ],
                            outputs=save_status
                        )

            # =================================================
            # EXPORT PDF
            # =================================================
            gr.Markdown("### 📄 Export")

            export_pdf_btn = gr.Button(
                "📄 Exporter la checklist en PDF"
            )

            pdf_file = gr.File(
                label="Checklist PDF"
            )
            export_pdf_btn.click(
                fn=export_checklist_pdf,
                inputs=[
                    checklist_state
                ],
                outputs=[
                    pdf_file
                ]
            )
            # =================================================
            # ANALYSE
            # =================================================
            gr.Markdown("### 🔍 Analyse de la checklist")

            analyze_btn = gr.Button(
                "🔍 Analyser checklist",
                variant="primary"
            )

            analysis_box = gr.Markdown()
            # -------------------------------------------------
            # Analyse basée sur checklist_id
            # -------------------------------------------------


            analyze_btn.click(
                fn=analyze_checklist_ui,
                inputs=[
                    checklist_state
                ],
                outputs=[
                    analysis_box
                ]
            )

            gr.Markdown("### 📑 Rapport d'audit")
            generate_report_btn = gr.Button("📑 Générer rapport",variant="primary")
            report_pdf = gr.File(label="Rapport PDF")
            generate_report_btn.click(fn=generate_report_ui,inputs=[checklist_state],outputs=[report_pdf])
            
    # ========================================================
    # BOUTON GENERATION
    # ========================================================

    generate_button.click(
        generate_checklist,
        inputs=[
            type_audit,
            site_audit
        ],
        outputs=[
            generation_status,
            checklist_state
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
